In [ ]:
# Colab: Python cell
!pip -q install rpy2 pandas==2.2.2 matplotlib

# Enable R in Colab
import os, warnings
os.environ["R_HOME"] = "/usr/lib/R"


In [ ]:
# Colab: Python cell – install R packages used by WGCNA
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
ro.r('''
options(warn=-1)
if (!require("BiocManager", quietly=TRUE)) install.packages("BiocManager", repos="https://cloud.r-project.org")
packs <- c("WGCNA","flashClust","dynamicTreeCut","pheatmap","VennDiagram","ggplot2","data.table")
for (p in packs) {
  if (!require(p, character.only=TRUE)) {
    if (p %in% rownames(installed.packages())) next
    tryCatch({
      BiocManager::install(p, ask=FALSE, update=FALSE)
    }, error=function(e) install.packages(p, repos="https://cloud.r-project.org"))
    library(p, character.only=TRUE)
  }
}
options(stringsAsFactors=FALSE)
allowWGCNAThreads()
''')


In [ ]:
# Colab: Python cell
import pandas as pd
import numpy as np

LABEL_COL = "label"   # <-- change if your phenotype column is named differently

COMMON_GENES = [
"TDRD9","OLFM4","SYTL2","PML","BTN3A2","PLXNA2","EIF4B","CYP2U1","IGF1R",
"GALNT11","HGF","DKKL1","GFI1B","PTMA","AHI1","SCAPER","SLC22A14","DOCK4","STAT2"
]

HUB_TABLE = pd.DataFrame({
    "Name":["IGF1R","STAT2","PML","HGF","HDAC1","JAK2","JAK1","MDM2","ERBB2","STAT3"],
    "Score":[59,21,13,9,2,2,2,2,2,2]
})

def load_matrix(path):
    """
    Reads a CSV where rows=samples and columns=genes (+ one label column).
    Returns (X, y), where:
      X: dataframe (samples x genes), numeric
      y: Series with 0/1 trait
    """
    df = pd.read_csv(path)
    # Try to locate label column robustly
    labcol = LABEL_COL if LABEL_COL in df.columns else (
        "Label" if "Label" in df.columns else
        "class" if "class" in df.columns else
        "Class" if "Class" in df.columns else None
    )
    if labcol is None:
        raise ValueError("Couldn't find a label column. Please set LABEL_COL to your label name.")
    y_raw = df[labcol].astype(str)
    # Map to 0/1 automatically
    if set(y_raw.unique()) <= {"0","1"}:
        y = y_raw.astype(int)
    else:
        # map first unique to 0, second to 1 (Control/Case)
        u = list(y_raw.unique())
        mapping = {u[0]:0, u[1]:1}
        y = y_raw.map(mapping)
    X = df.drop(columns=[labcol]).copy()
    # Keep only numeric
    X = X.apply(pd.to_numeric, errors="coerce")
    # Remove all-NA cols
    X = X.dropna(axis=1, how="all")
    # Fill residual NAs with column medians
    X = X.apply(lambda c: c.fillna(c.median()), axis=0)
    return X, y

def to_r_df(df, name):
    """Send a pandas dataframe to R with the given symbol name."""
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()
    ro.globalenv[name] = pandas2ri.py2rpy(df)


In [ ]:
# Colab: Python cell
import textwrap
import rpy2.robjects as ro

WGCNA_R = ro.r("""
run_wgcna <- function(expr_df, trait_vec, dataset_name="dataset") {
  suppressPackageStartupMessages({
    library(WGCNA); library(dynamicTreeCut); library(flashClust)
    library(pheatmap); library(VennDiagram); library(ggplot2)
  })
  options(stringsAsFactors=FALSE)
  datExpr <- expr_df   # rows=samples, cols=genes

  # === A) Sample clustering dendrogram ===
  pdf(paste0(dataset_name,"_A_sample_dendrogram.pdf"), width=10, height=6)
  sampleTree <- hclust(dist(datExpr), method="average")
  par(cex=0.6); plot(sampleTree, main=paste(dataset_name,"- Sample clustering"), sub="", xlab="")
  dev.off()

  # === C) pickSoftThreshold plots ===
  powers = c(c(1:10), seq(from=12, to=30, by=2))
  sft <- pickSoftThreshold(datExpr, powerVector=powers, verbose=5, networkType="signed")
  chosenPower <- ifelse(is.na(sft$powerEstimate), 6, sft$powerEstimate)
  pdf(paste0(dataset_name,"_C_softThreshold.pdf"), width=12, height=5)
  par(mfrow=c(1,2))
  plot(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
       xlab="Soft Threshold (power)", ylab="Scale Free Topology Model Fit, signed R^2",
       type="n", main="Scale independence")
  text(sft$fitIndices[,1], -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
       labels=powers, cex=0.9)
  abline(h=0.8, col="red")
  plot(sft$fitIndices[,1], sft$fitIndices[,5],
       xlab="Soft Threshold (power)", ylab="Mean connectivity", type="n", main="Mean connectivity")
  text(sft$fitIndices[,1], sft$fitIndices[,5], labels=powers, cex=0.9)
  dev.off()

  # === B) Network construction + module detection ===
  net <- blockwiseModules(
    datExpr, power=chosenPower, TOMType="signed", minModuleSize=30,
    reassignThreshold=0, mergeCutHeight=0.25, numericLabels=TRUE, pamRespectsDendro=FALSE,
    saveTOMs=FALSE, verbose=3
  )
  MEs <- net$MEs; moduleLabels <- net$colors; moduleColors <- labels2colors(moduleLabels)
  geneTree <- net$dendrograms[[1]]

  pdf(paste0(dataset_name,"_B_gene_dendro_modules.pdf"), width=12, height=6)
  plotDendroAndColors(geneTree, moduleColors[net$blockGenes[[1]]],
                      "Module colors", dendroLabels=FALSE, hang=0.03,
                      addGuide=TRUE, guideHang=0.05, main=paste(dataset_name,"- Gene dendrogram & modules"))
  dev.off()

  # === D) Module–trait relationships ===
  datTraits <- data.frame(Trait=as.numeric(trait_vec)) # 0/1
  MEs0 <- orderMEs(MEs)
  modTraitCor <- cor(MEs0, datTraits$Trait, use="p")
  modTraitP   <- corPvalueStudent(modTraitCor, nrow(datExpr))

  # heatmap
  textMatrix <- paste(signif(modTraitCor,2), "\n(", signif(modTraitP,1), ")", sep="")
  dim(textMatrix) <- dim(modTraitCor)
  pdf(paste0(dataset_name,"_D_module_trait_heatmap.pdf"), width=12, height=6)
  labeledHeatmap(Matrix=t(modTraitCor),
                 yLabels=names(MEs0), xLabels="Trait (0/1)",
                 ySymbols=names(MEs0), colorLabels=FALSE, colors=blueWhiteRed(50),
                 textMatrix=t(textMatrix), setStdMargins=FALSE, cex.text=0.8,
                 zlim=c(-1,1), main=paste(dataset_name,"- Module–trait relationships"))
  dev.off()

  # Collect outputs for Python side
  res <- list(
    chosenPower=chosenPower,
    moduleColors=moduleColors,
    moduleLabels=moduleLabels,
    geneNames=colnames(datExpr),
    kWithin = intramodularConnectivity(
      adjacency(datExpr, power=chosenPower, type="signed"),
      moduleColors
    )$kWithin
  )
  return(res)
}
""")


In [ ]:
pip install rpy2


In [ ]:
%load_ext rpy2.ipython


In [ ]:
%%R
install.packages("BiocManager", repos = "https://cloud.r-project.org")
BiocManager::install("limma", ask = FALSE)
BiocManager::install("WGCNA", ask = FALSE)


In [ ]:
%%R
library(limma)
library(WGCNA)


In [ ]:
%%R
options(stringsAsFactors = FALSE)

req_pkgs <- c("WGCNA","data.table","ggplot2","pheatmap","VennDiagram","limma","pROC","grid")
for (p in req_pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages({
  library(WGCNA); library(data.table); library(ggplot2); library(pheatmap)
  library(VennDiagram); library(limma); library(pROC); library(grid)
})
allowWGCNAThreads()

OUTDIR <- "WGCNA_outputs"; if (!dir.exists(OUTDIR)) dir.create(OUTDIR)

Top10_list <- c("IGF1R","STAT2","PML","HGF","HDAC1","JAK2","JAK1","MDM2","ERBB2","STAT3")


In [ ]:
%%R
TRAIN_EXPR <- "/content/GSE63878_expression.csv"
TRAIN_LABEL<- "/content/GSE63878_labels.csv"
TEST_EXPR  <- "/content/GSE98793_expression.csv"
TEST_LABEL <- "/content/GSE98793_labels.csv"

read_expr <- function(path){
  dt <- fread(path)
  rn <- dt[[1]]; dt[[1]] <- NULL
  m <- as.data.frame(dt); rownames(m) <- make.unique(rn)
  m
}
read_labels <- function(path){
  lab <- fread(path)
  cn <- tolower(names(lab))
  id_col  <- if (any(cn %in% c("sample","sampleid","id","geo_accession","gsm","name"))) which(cn %in% c("sample","sampleid","id","geo_accession","gsm","name"))[1] else 1
  grp_col <- if (any(cn %in% c("group","status","phenotype","class","label","casecontrol","condition"))) which(cn %in% c("group","status","phenotype","class","label","casecontrol","condition"))[1] else 2
  df <- data.frame(Sample = as.character(lab[[id_col]]), Group = lab[[grp_col]])
  g <- tolower(as.character(df$Group))
  if (all(g %in% c("case","control"))) df$Group <- ifelse(g=="case",1,0)
  else {
    sup <- suppressWarnings(as.numeric(g))
    df$Group <- if (all(!is.na(sup))) sup else as.numeric(factor(g)) - 1
  }
  df$Group <- as.numeric(df$Group); df
}
expr_train <- read_expr(TRAIN_EXPR)
lab_train  <- read_labels(TRAIN_LABEL)
expr_test  <- read_expr(TEST_EXPR)
lab_test   <- read_labels(TEST_LABEL)

align_by_labels <- function(expr, lab){
  common <- intersect(colnames(expr), lab$Sample)
  if (length(common) < 10) stop("Too few overlapping samples. Check sample IDs.")
  list(expr = expr[, common, drop=FALSE],
       lab  = lab[match(common, lab$Sample), , drop=FALSE])
}
al_tr <- align_by_labels(expr_train, lab_train); expr_train <- al_tr$expr; lab_train <- al_tr$lab
al_te <- align_by_labels(expr_test,  lab_test);  expr_test  <- al_te$expr; lab_test  <- al_te$lab


In [ ]:
%%R
datExpr_train <- as.data.frame(t(expr_train))
rownames(datExpr_train) <- colnames(expr_train)

gsg <- goodSamplesGenes(datExpr_train, verbose = 3)
if (!gsg$allOK) datExpr_train <- datExpr_train[gsg$goodSamples, gsg$goodGenes]

# Sample dendrogram
sampleTree <- hclust(dist(datExpr_train), method = "average")
plot(sampleTree, main="Sample clustering to detect outliers", xlab="", sub="")

# Save
png(file.path(OUTDIR,"A_sample_clustering.png"), width=1200, height=600)
plot(sampleTree, main="Sample clustering to detect outliers", xlab="", sub="")
dev.off()

jpeg(file.path(OUTDIR,"A_sample_clustering.jpg"), width=1200, height=600, quality=95)
plot(sampleTree, main="Sample clustering to detect outliers", xlab="", sub="")
dev.off()

pdf(file.path(OUTDIR,"A_sample_clustering.pdf"), width=12, height=6)
plot(sampleTree, main="Sample clustering to detect outliers", xlab="", sub="")
dev.off()


In [ ]:
%%R
powers <- 1:20
sft <- pickSoftThreshold(datExpr_train, powerVector = powers, networkType="signed", verbose=5)

# PNG save
png(file.path(OUTDIR,"C_soft_threshold.png"), width=1600, height=800, res=150)
par(mfrow=c(1,2))

# ---- Scale independence ----
plot(sft$fitIndices[,1],
     -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     xlab="Soft Threshold (power)",
     ylab="Scale Free Topology Model Fit, signed R^2",
     main="Scale independence",
     type="b", pch=19, col="red")   # point + line
abline(h=0.8, col="blue", lty=2)

text(sft$fitIndices[,1],
     -sign(sft$fitIndices[,3])*sft$fitIndices[,2],
     labels=powers, col="black", cex=1.2, pos=3)

# ---- Mean connectivity ----
plot(sft$fitIndices[,1],
     sft$fitIndices[,5],
     xlab="Soft Threshold (power)",
     ylab="Mean connectivity",
     main="Mean connectivity",
     type="b", pch=19, col="red")   # point + line

text(sft$fitIndices[,1],
     sft$fitIndices[,5],
     labels=powers, col="black", cex=1.2, pos=3)

dev.off()

# Final chosen soft threshold
softPower <- if (!is.na(sft$powerEstimate)) sft$powerEstimate else 6
softPower


In [ ]:
%%R
net <- blockwiseModules(
  datExpr_train,
  power = softPower,
  TOMType = "signed",
  networkType = "signed",
  minModuleSize = 30,
  reassignThreshold = 0,
  mergeCutHeight = 0.25,
  numericLabels = TRUE,
  pamRespectsDendro = FALSE,
  saveTOMs = TRUE,
  saveTOMFileBase = file.path(OUTDIR, "trainTOM"),
  verbose = 3
)

moduleColors <- labels2colors(net$colors)

# Dendrogram + colors
plotDendroAndColors(net$dendrograms[[1]],
                    moduleColors[net$blockGenes[[1]]],
                    "Module colors",
                    dendroLabels = FALSE, hang = 0.03,
                    addGuide = TRUE, guideHang = 0.05,
                    main = "Cluster Dendrogram")

# PNG
png(file.path(OUTDIR,"B_cluster_dendrogram.png"), width=1400, height=700, res=150)
plotDendroAndColors(net$dendrograms[[1]],
                    moduleColors[net$blockGenes[[1]]],
                    "Module colors",
                    dendroLabels = FALSE, hang = 0.03,
                    addGuide = TRUE, guideHang = 0.05,
                    main = "Cluster Dendrogram")
dev.off()

# JPEG
jpeg(file.path(OUTDIR,"B_cluster_dendrogram.jpg"), width=1400, height=700, quality=95)
plotDendroAndColors(net$dendrograms[[1]],
                    moduleColors[net$blockGenes[[1]]],
                    "Module colors",
                    dendroLabels = FALSE, hang = 0.03,
                    addGuide = TRUE, guideHang = 0.05,
                    main = "Cluster Dendrogram")
dev.off()

# PDF
pdf(file.path(OUTDIR,"B_cluster_dendrogram.pdf"), width=14, height=7)
plotDendroAndColors(net$dendrograms[[1]],
                    moduleColors[net$blockGenes[[1]]],
                    "Module colors",
                    dendroLabels = FALSE, hang = 0.03,
                    addGuide = TRUE, guideHang = 0.05,
                    main = "Cluster Dendrogram")
dev.off()


In [ ]:
%%R
# ==========================
# Load Required Libraries
# ==========================
library(limma)
library(WGCNA)
library(VennDiagram)
library(grid)

# ==========================
# Step 1. Define Gene Sets
# ==========================
# Hub Genes (exactly 10)
hub_genes <- c("IGF1R","STAT2","PML","HGF","HDAC1",
               "JAK2","JAK1","MDM2","ERBB2","STAT3")

# Common Genes (exactly 19)
common_genes <- c("TDRD9","OLFM4","SYTL2","PML","BTN3A2","PLXNA2",
                  "EIF4B","CYP2U1","IGF1R","GALNT11","HGF","DKKL1",
                  "GFI1B","PTMA","AHI1","SCAPER","SLC22A14","DOCK4","STAT2")

# WGCNA Genes – আপাতত placeholder (তোমার analysis থেকে replace করবে)
WGCNA_genes <- c("IGF1R","STAT3","ERBB2","PML","MDM2",
                 "GALNT11","PLXNA2","SYTL2","DOCK4","HDAC1")

# ==========================
# Step 2. Create Venn Diagram
# ==========================
venn.plot <- venn.diagram(
  x = list(
    Stress   = common_genes,   # এখানে common gene কে SS ধরা হলো
    Depression = WGCNA_genes,    # placeholder
    #Deprssion = common_genes,
    WGCNA= hub_genes       # hub gene কে WGCNA group ধরা হলো
  ),
  filename = NULL,
  fill = c("red", "blue", "green"),
  alpha = 0.5,
  cex = 2,
  cat.cex = 2,
  main = "Venn Diagram: Stress vs Depression vs WGCNA"
)

png("SS_NASH_WGCNA_venn.png", width = 2000, height = 2000, res = 300)
grid.draw(venn.plot)
dev.off()

cat("Venn diagram saved as SS_NASH_WGCNA_venn.png\n")

# ==========================
# Step 3. Sanity Check Counts
# ==========================
cat("\nGene set sizes:\n")
cat("SS (common genes):", length(common_genes), "\n")
cat("NASH (WGCNA placeholder):", length(WGCNA_genes), "\n")
cat("WGCNA (hub genes):", length(hub_genes), "\n")


In [ ]:
%%R
# ===============================
# Load Required Libraries
# ===============================
library(pheatmap)
library(RColorBrewer)

# ===============================
# 1. Define Top 10 Hub Genes
# ===============================
top_genes <- c("IGF1R", "STAT2", "PML", "HGF", "HDAC1",
               "JAK2", "JAK1", "MDM2", "ERBB2", "STAT3")

# ===============================
# 2. Create Dummy Expression Data (10 genes x 10 samples)
# ===============================
set.seed(123)
expr_matrix <- matrix(rnorm(100, mean=5, sd=2), nrow=10, ncol=10)
rownames(expr_matrix) <- top_genes
colnames(expr_matrix) <- c("MEYellow","MEPurple","MEBlue","MEGreen","MERed",
                           "MEBrown","METurquoise","MEMagenta","MEBlack","MEGray")

# ===============================
# 3. Save Simple Heatmap as PNG
# ===============================
output_file <- "/content/Top10HubGenes_PlaneHeatmap.png"

png(filename = output_file, width = 1200, height = 1000, res = 150)
pheatmap(expr_matrix,
         cluster_rows = FALSE,   # no row clustering
         cluster_cols = FALSE,   # no column clustering
         show_rownames = TRUE,
         show_colnames = TRUE,
         fontsize_row = 12,
         fontsize_col = 12,
         main = "Top 10 Hub Genes Plane Heatmap",
         color = colorRampPalette(c("blue", "white", "red"))(50),
         border_color = "black")
dev.off()

# ===============================
# 4. Print PNG path
# ===============================
output_file
